# sleep-analyze 学習ランナー (Colab + Cloudflare R2)

左側の「シークレット（鍵アイコン）」に以下を登録し、ノートブックからのアクセスを有効にしてから実行してください。

- **必須**: `R2_ENDPOINT_URL`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET`
- **任意**: `R2_TRAIN_PREFIX` (既定: `train`), `DISCORD_WEBHOOK_URL`

In [ ]:
import os
import shutil
import subprocess

REPO_DIR = "/content/sleep-analyze"
REPO_URL = "https://github.com/Nlkomaru/sleep-analyze.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

local_bin = os.path.expanduser("~/.local/bin")
if local_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = local_bin + os.pathsep + os.environ.get("PATH", "")

if shutil.which("uv") is None:
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)

!uv sync --frozen
!uv run python -c "import torch; print('torch=', torch.__version__, 'cuda_available=', torch.cuda.is_available(), 'cuda=', torch.version.cuda)"

In [ ]:
import os
from google.colab import userdata

REQUIRED_SECRETS = [
    "R2_ENDPOINT_URL",
    "R2_ACCESS_KEY_ID",
    "R2_SECRET_ACCESS_KEY",
    "R2_BUCKET",
]
OPTIONAL_SECRETS = [
    "R2_TRAIN_PREFIX",
    "DISCORD_WEBHOOK_URL",
]

for key in REQUIRED_SECRETS:
    val = userdata.get(key)
    if not val:
        raise RuntimeError(f"Missing Colab Secret: {key}")
    os.environ[key] = val

for key in OPTIONAL_SECRETS:
    try:
        val = userdata.get(key)
    except Exception:
        val = ""
    if val:
        os.environ[key] = val

!uv run python train/src/r2_store.py check

In [ ]:
import os

MODE = "refresh" #@param ["hpo", "refresh"]
RUN_NAME = "convnext-2026-08-01" #@param {type:"string"}
INIT_CHECKPOINT_KEY = "" #@param {type:"string"}

os.environ["MODE"] = MODE
os.environ["RUN_NAME"] = RUN_NAME
os.environ["INIT_CHECKPOINT_KEY"] = INIT_CHECKPOINT_KEY.strip()
print(f"MODE={MODE} RUN_NAME={RUN_NAME} INIT_CHECKPOINT_KEY={os.environ['INIT_CHECKPOINT_KEY'] or '(none)'}")

In [ ]:
import os

TB_LOGDIR = f"train/build/runs/{os.environ['RUN_NAME']}/tensorboard"
os.makedirs(TB_LOGDIR, exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir {TB_LOGDIR}

In [ ]:
%%bash
set -euo pipefail
ARGS=(--mode "$MODE" --run-name "$RUN_NAME")
if [[ -n "${INIT_CHECKPOINT_KEY:-}" ]]; then
  ARGS+=(--init-checkpoint-key "$INIT_CHECKPOINT_KEY")
fi
uv run python -u train/src/sweep.py "${ARGS[@]}"

In [ ]:
# オプション: Colab 上でそのままアンサンブル化する（現本番モデル比較なし）
!uv run python train/src/finalize_ensemble.py --run-name "$RUN_NAME" --no-baseline